In [1]:
import sys
import os

import itertools
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm as tqdm_auto
from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader, Dataset
import pytorch_lightning as pl
from IPython.display import clear_output


mpl.rcParams['pdf.fonttype'] = 42

/mnt/calc/skaraban_smtb/.conda/envs/utrgan/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

from utilities.pl_regressor import RNARegressor
from utilities.UTR5_utilities import predict, seqprep, Condition2Tensor, UTRData, make_res


In [3]:
from utilities.legnet_softclass import LegNetClassifier
import utilities.legnet_softclass as legnet_classifier
sys.modules['legnet_classifier'] = legnet_classifier


In [4]:
def seed_everything(seed):
    # random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    
seed_everything(42)

In [5]:
device = 'cuda:3'
num_workers = 16

batch_size = 64
seq_len = 50
CELL_TYPE_FILTER = 'c2'

CELLTYPE_CODES_UTR3 = {"c1": 0,
                       "c2": 1,
                       "c4": 2,
                       "c6": 3,
                       "c17": 4,
                       "c13": 5,
                       "c10": 6}

CELLTYPE_UTR3 = {"c1": 0,
                       "c2": 1,
                       "c4": 2,
                       "c6": 3,
                       "c17": 4,
                       "c13": 5}

CELLTYPE_CODES_UTR5 = {"c1": 0,
                       "c2": 1,
                       "c4": 2,
                       "c6": 3,
                       "c17": 4}


c2t = Condition2Tensor(num_conditions=len(CELLTYPE_CODES_UTR5), celltype_codes=CELLTYPE_CODES_UTR5)

In [6]:
model_path = f"utilities/epoch=14-step=2520.ckpt"
device = 'cuda:2'

model_predict = RNARegressor.load_from_checkpoint(model_path, map_location='cpu').model.to(device)
model_predict.eval()
input_len = 50

In [7]:
def predict(model_predict, dataset, batch_size=64):
    model_predict.eval()
    pred_array = []
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for batch in tqdm(dataloader):
            res = model_predict(batch, softmax=True).cpu().detach()
            tmp = (res * torch.tensor([1,2,3,4])).sum(dim=1)
            pred_array.append(tmp)
    return pred_array

scoring 

In [ ]:

from tqdm import tqdm

in_file = 'results/gan_generated_sequences_UTR5_16dim_1M.csv'
cell_lines = ['c1', 'c2', 'c4', 'c6', 'c17']
CHUNK_SIZE = 1_000_000  
BATCH_SIZE = 1024      
MAX_SAMPLES = 1_000_000

df_full = pd.read_csv(in_file).iloc[:MAX_SAMPLES]

for cell_line in cell_lines:
    out_file_csv = f'data/scored_UTR5_16dim_1M_{cell_line}.csv'
    
    if os.path.exists(out_file_csv):
        os.remove(out_file_csv)
    
    first = True
    num_chunks = (len(df_full) + CHUNK_SIZE - 1) // CHUNK_SIZE

    with tqdm(total=num_chunks, desc=f"{cell_line}") as pbar:
        for start in range(0, len(df_full), CHUNK_SIZE):
            end = min(start + CHUNK_SIZE, len(df_full))
            chunk = df_full.iloc[start:end].copy()

            scores_all = []
            for b_start in range(0, len(chunk), BATCH_SIZE):
                b_end = min(b_start + BATCH_SIZE, len(chunk))
                batch = chunk.iloc[b_start:b_end]

                seq_logits = [seqprep(seq) for seq in batch.sequence]
                seq_logits = torch.concat(seq_logits)

                preds = predict(seq_logits, model_predict, cell_line, device=device)
                scores_all.extend(preds)

                pbar.update(1)

            chunk["score"] = scores_all
            chunk.to_csv(out_file_csv, mode="a", header=first, index=False)
            first = False  

    print(f"[+] Scoring saved to {out_file_csv}")

In [8]:

from tqdm import tqdm

in_file = 'data/gan_generated_sequences_UTR5_10k.csv'
cell_lines = ['c1', 'c2', 'c4', 'c6', 'c17']
CHUNK_SIZE = 1_000_000  
BATCH_SIZE = 1024      
MAX_SAMPLES = 1_000_000

for cell_line in cell_lines:
    first = True
    df = pd.read_csv(in_file).iloc[:MAX_SAMPLES]
    out_file_csv = f'data/scored_UTR5_16dim_10k_{cell_line}.csv'
    
    if os.path.exists(out_file_csv):
        os.remove(out_file_csv)
        
    num_chunks = (len(df) + CHUNK_SIZE - 1) // CHUNK_SIZE

    for i in range(num_chunks):
        start = i * CHUNK_SIZE
        end = min((i + 1) * CHUNK_SIZE, len(df))
        batch = df.iloc[start:end].copy()


        ds = UTRData(seqs=batch.sequence, cell_type=cell_line, UTR='5UTR', device=device)
        preds = predict(model_predict, ds, batch_size=BATCH_SIZE)
        scores = torch.concat(preds).cpu().numpy()
        batch = batch.copy()
        batch["score"] = scores

        write_header = not os.path.exists(out_file_csv)
        
        batch.to_csv(
            out_file_csv, 
            mode="a", 
            header=write_header, 
            index=False
        )
        first = False  

    print(f"[+] Scoring saved to {out_file_csv}")


  0%|          | 0/10 [00:00<?, ?it/s]100%|██████████| 10/10 [00:31<00:00,  3.16s/it]


[+] Scoring saved to data/scored_UTR5_16dim_10k_c1.csv


100%|██████████| 10/10 [00:16<00:00,  1.70s/it]


[+] Scoring saved to data/scored_UTR5_16dim_10k_c2.csv


100%|██████████| 10/10 [00:17<00:00,  1.70s/it]


[+] Scoring saved to data/scored_UTR5_16dim_10k_c4.csv


100%|██████████| 10/10 [00:17<00:00,  1.70s/it]


[+] Scoring saved to data/scored_UTR5_16dim_10k_c6.csv


100%|██████████| 10/10 [00:17<00:00,  1.70s/it]

[+] Scoring saved to data/scored_UTR5_16dim_10k_c17.csv
